In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import os

In [ ]:

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def add_user_message(messages, text):
    user_message = {"role":"user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    asst_message = {"role":"assistant", "content": text}
    messages.append(asst_message)

def chat(messages):
    resposne = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens = 200,
        messages=messages,
    )
    return resposne.content[0].text

### Multi-turn conversation

In [ ]:
messages = []
add_user_message(messages, "What is the capital of France? in one word.")
response1 = chat(messages)
print(response1)
add_assistant_message(messages, response1)
add_user_message(messages, "What is the population of that city?")
response2 = chat(messages)
print(response2)
add_assistant_message(messages, response2)

## System Prompts

Additional description of claude's role given so it anwers in a particular manner and tone. 

In [ ]:
##example

system_prompts ='''
You are a patient math tutor that helps students understand math concepts. You provide clear explanations and step-by-step solutions to problems.
'''
response = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens = 200,
        messages=messages,
        system=system_prompts
    )

## System prompts cannoty be none if you decide to use them, so use two variants of the call with and without the system argument if you wanna make it optional.

NameError: name 'client' is not defined

## Temperature (0<x<1)

Low temp = More deterministic output. Selects output with the highest initital probability

    --Good for factual and data driven tasks

High temp = More random output. More random and creative

    --Good for brainstorming or marketing tasks

In [ ]:
## example usage of TEMPERATURE

system_prompts ='''
You are a patient math tutor that helps students understand math concepts. You provide clear explanations and step-by-step solutions to problems.
'''
response = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens = 200,
        messages=messages,
        system=system_prompts,
        temperature=0.9
    )

print(response.content[0].text)


## Response Streaming
Stream token by token or word by word insteading of waiting for the full response from Claude.

In [ ]:
## example usage

response = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens = 200,
        messages=messages,
        temperature=0.9,
        stream=True
    )

for chunk in response:
    print(chunk)

watch out for the event : RawContentBlockDeltaEvent - Chunk related to the latest content for which generation was started.

{
    MessageStart 

    ContentBlockStart 

    ContentBlockDelta

    ContentBlockDelta 

    ContentBlockDelta 

    ContentBlockStopv 

    MessageDelta 
    
    Messagestop
}

In [ ]:
## easier solution that filering out the event types every time: 
with client.messages.stream(
    model="claude-sonnet-4-0",
    max_tokens = 200,
    messages=messages,
    temperature=0.9
) as stream:
    for text in stream.text_stream:
        print(text, end="")



## we can go further by:
with client.messages.stream(
    model="claude-sonnet-4-0",
    max_tokens = 200,
    messages=messages,
    temperature=0.9
) as stream:
    for text in stream.text_stream:
        #print(text, end="")
        pass
stream.get_final_message()      ## to store in a list for conversation history and conteual mutli-turn conversations.

## Structured data
Tells claude what to generate as structured outptut. Example code geenration, bullet lists etc. To get raw outptut that can be straight up copied and used. 

We do this by using stop sequences. For example if claude starts geenrating a json with ``` bla bla { bla}

stop sequence = " ``` " so claude will now skip that sequence and thinks its already generated it. 

you can later use json.loads(text.strip()) to parse the output as a json or use the Markdown library to parse as markdown. 

In [ ]:
## example usage

add_user_message(messages, "give a general AWS event bridge rule as json")
add_assistant_message(messages, "here is the json without any additional comments. /n ```json") ## This tells claude its output should start with this 
response = client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens = 200,
        messages=messages,
        system=system_prompts,
        temperature=0.9,
        stop_sequences=["```"]
    )

### Look at you, all good to go with the Claude API